## Post-processing

Extract unique drug names and provider names from the pipeline results, then build normalization maps.

In [2]:
import json
from pathlib import Path
from collections import Counter

results = json.loads(Path("../data/results.json").read_text())
records = results["records"]
print(f"Loaded {len(records)} records")

Loaded 50 records


### Drug / treatment names

In [3]:
drug_counter = Counter()
for record in records:
    for treatment in record["call1"]["treatment_journey"]:
        drug_counter[treatment["name"].strip().lower()] += 1

print(f"{len(drug_counter)} unique drug names (by frequency):\n")
for name, count in drug_counter.most_common():
    print(f"  {count:>3}x  {name}")

25 unique drug names (by frequency):

   40x  mesalamine
   34x  prednisone
   29x  methotrexate
   27x  humira
   21x  remicade
   21x  stelara
   19x  azathioprine
    9x  entyvio
    8x  sulfasalazine
    8x  biologics
    7x  budesonide
    2x  6-mp
    2x  nsaids
    1x  iron supplements
    1x  tramadol
    1x  hydroxychloroquine
    1x  antispasmodics
    1x  cimzia
    1x  fiber supplements
    1x  cymbalta
    1x  lamotrigine
    1x  ursodiol
    1x  clinical trial drug
    1x  folic acid
    1x  antibiotics


### Provider / pathway step names

In [4]:
provider_counter = Counter()
for record in records:
    for step in record["call3"]["referral_pathway"]:
        provider_counter[step["name"].strip().lower()] += 1

print(f"{len(provider_counter)} unique provider names (by frequency):\n")
for name, count in provider_counter.most_common():
    print(f"  {count:>3}x  {name}")

31 unique provider names (by frequency):

   29x  gastroenterologist
   10x  doctor
    7x  general practitioner
    4x  gi doctor
    3x  er
    2x  emergency room
    2x  health center
    2x  new doctor
    2x  gi
    1x  primary care doctor
    1x  endocrinologist
    1x  multiple doctors
    1x  university health center
    1x  primary care
    1x  family doctor
    1x  gi specialist
    1x  er physician
    1x  campus health
    1x  primary doctor
    1x  emergency room physician
    1x  psychiatrist
    1x  third doctor
    1x  pediatric gi
    1x  ob
    1x  primary
    1x  rheumatologist
    1x  hospital
    1x  three doctors
    1x  general physician
    1x  general practitioner or non-specialist
    1x  surgeon


### Mapping coverage check

In [7]:
def check_mapping_coverage(counter: Counter, mapping: dict, label: str) -> None:
    in_data = set(counter.keys())
    in_map  = set(mapping.keys())

    missing = in_data - in_map   # in data, not in map → lookup will fail
    unused  = in_map  - in_data  # in map, never seen in data → dead entry

    print(f"=== {label} ===")
    print(f"  Unique names in data : {len(in_data)}")
    print(f"  Keys in mapping      : {len(in_map)}")

    if missing:
        print(f"\n  ❌ {len(missing)} names in data with NO mapping (lookup will return None):")
        for name in sorted(missing):
            print(f"     {counter[name]:>3}x  {name!r}")
    else:
        print(f"\n  ✅ All data names are covered by the mapping")

    if unused:
        print(f"\n  ⚠️  {len(unused)} mapping keys never seen in data (dead entries):")
        for name in sorted(unused):
            print(f"       {name!r}")
    else:
        print(f"  ✅ No dead entries in the mapping")


drug_map     = json.loads(Path("../data/crohns_drug_mapping.json").read_text())
provider_map = json.loads(Path("../data/crohns_provider_mapping.json").read_text())

check_mapping_coverage(drug_counter,     drug_map,     "Drugs")
print()
check_mapping_coverage(provider_counter, provider_map, "Providers")

=== Drugs ===
  Unique names in data : 25
  Keys in mapping      : 25

  ✅ All data names are covered by the mapping
  ✅ No dead entries in the mapping

=== Providers ===
  Unique names in data : 31
  Keys in mapping      : 73

  ✅ All data names are covered by the mapping

  ⚠️  42 mapping keys never seen in data (dead entries):
       'Dr. Anderson'
       'Dr. Williams'
       'ER physician'
       'Emergency Room physician'
       'General Physician'
       'General practitioner or non-specialist'
       'cardiologist'
       'dermatologist'
       'dr. anderson, my gi'
       'dr. martinez'
       'dr. rebecca chen'
       "family friend who's a gi"
       'female gi doctor'
       'first gi'
       'gastroenterologist (dr. patel)'
       "gastroenterologist (implied by colonoscopy and crohn's treatment)"
       'general doctor'
       'general doctor/physician'
       'general doctors'
       "general practitioner / primary care (implied by 'doctors' and initial ibs diagnosis)"
 